In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install torch torchvision tqdm flask flask-cors pillow

In [4]:
import os
print(os.listdir('/content/drive/MyDrive/PLANTVILLAGE'))

['Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato__Target_Spot', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato_Leaf_Mold', 'Tomato_Late_blight', 'Tomato_healthy', 'Tomato_Early_blight', 'Tomato_Bacterial_spot', 'Potato___Late_blight', 'Potato___healthy', 'Potato___Early_blight', 'Pepper__bell___healthy', 'Pepper__bell___Bacterial_spot']


In [5]:
import os, json, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# CONFIG
DATA_DIR    = "/content/drive/MyDrive/PLANTVILLAGE"
MODEL_SAVE  = "plant_disease_efficientnet.pth"
CLASS_JSON  = "class_names.json"
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 1e-4
WEIGHT_DECAY= 1e-5
VAL_SPLIT   = 0.15
NUM_WORKERS = 2
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# TRANSFORMS
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# DATASET
full_dataset = datasets.ImageFolder(DATA_DIR)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print(f"Classes found: {num_classes} → {class_names}")

with open(CLASS_JSON, "w") as f:
    json.dump(class_names, f, indent=2)

val_size   = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
train_ds.dataset.transform = train_transform
val_ds.dataset.transform   = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# MODEL
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, num_classes),
)
model = model.to(DEVICE)

# TRAINING
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc = 0.0
for epoch in range(1, EPOCHS+1):
    # Train
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} Train"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total   += imgs.size(0)

    # Validate
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} Val  "):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total   += imgs.size(0)

    scheduler.step()
    train_acc = t_correct / t_total
    val_acc   = v_correct / v_total
    tag = " ← BEST" if val_acc > best_val_acc else ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "num_classes": num_classes}, MODEL_SAVE)

    print(f"Epoch {epoch:02d}/{EPOCHS} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}{tag}")

print(f"\n✅ Training complete! Best val accuracy: {best_val_acc:.4f}")

Using device: cuda
Classes found: 15 → ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 120MB/s]
Epoch 1/10 Val  : 100%|██████████| 97/97 [08:08<00:00,  5.03s/it]


Epoch 01/10 | train_acc=0.8723 | val_acc=0.9887 ← BEST


Epoch 2/10 Val  : 100%|██████████| 97/97 [00:18<00:00,  5.38it/s]


Epoch 02/10 | train_acc=0.9825 | val_acc=0.9929 ← BEST


...

Epoch 08/10 | train_acc=0.9981 | val_acc=0.9981 ← BEST


Epoch 9/10 Val  : 100%|██████████| 97/97 [00:17<00:00,  5.62it/s]


Epoch 09/10 | train_acc=0.9988 | val_acc=0.9974


Epoch 10/10 Val  : 100%|██████████| 97/97 [00:18<00:00,  5.29it/s]

Epoch 10/10 | train_acc=0.9989 | val_acc=0.9981

✅ Training complete! Best val accuracy: 0.9981


In [7]:
from google.colab import files
files.download('plant_disease_efficientnet.pth')
files.download('class_names.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>